In [1]:
from pyspark.sql import SparkSession

# Création de la session Spark
spark = SparkSession.builder \
    .appName("TP_RDD_Operations") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
print("Spark est prêt ! Version :", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/15 11:28:30 WARN Utils: Your hostname, codespaces-52671d, resolves to a loopback address: 127.0.0.1; using 10.0.1.179 instead (on interface eth0)
26/04/15 11:28:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 11:28:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark est prêt ! Version : 4.1.1


In [2]:
# Read back
df_final_spark = spark.read.parquet("final.parquet")
df_to_predict_spark = spark.read.parquet("to_predict.parquet")

In [3]:
# Number of rows
num_rows = df_final_spark.count()

# Number of columns
num_cols = len(df_final_spark.columns)

print(f"The final dataset contains {num_rows} lines and {num_cols} columns")

The final dataset contains 2945 lines and 11 columns


In [4]:
import re 
def top_k_words(df, column, k):
    # 1) MAP: extract text column
    rdd_lignes = df.select(column).rdd.map(lambda row: row[0])

    # 2) MAP: tokenize
    rdd_mots = rdd_lignes.flatMap(lambda ligne: ligne.split())

    # 3) MAP: clean words
    rdd_mots_propres = rdd_mots.map(
        lambda mot: re.sub(r"[^a-zA-Z0-9]", "", mot).lower()
    )

    # 4) FILTER: remove empty + short words
    rdd_mots_filtres = rdd_mots_propres.filter(lambda mot: mot != "" and len(mot) > 1)

    # 5) MAP: (word, 1)
    rdd_paires = rdd_mots_filtres.map(lambda mot: (mot, 1))

    # 6) REDUCE: count occurrences
    rdd_comptes = rdd_paires.reduceByKey(lambda a, b: a + b)

    # 7) TOP-K (MapReduce-style sorting step)
    top_k = rdd_comptes.takeOrdered(k, key=lambda x: -x[1])

    return top_k

In [5]:
top10 = top_k_words(df_final_spark, "description", 10)

print("Top 10 words:")
print(top10)

Top 10 words:
[('and', 25902), ('to', 12632), ('the', 12369), ('data', 11384), ('of', 8416), ('in', 6647), ('for', 6184), ('with', 5713), ('our', 4581), ('is', 4230)]


In [6]:
top10 = top_k_words(df_to_predict_spark, "description", 10)

print("Top 10 words:")
print(top10)

Top 10 words:
[('and', 6587), ('to', 2711), ('data', 2486), ('the', 2003), ('of', 1850), ('with', 1581), ('in', 1437), ('for', 1189), ('or', 1005), ('experience', 938)]
